Modèle GRU sur les 10 jours suivants 

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Input
from tabulate import tabulate
import tensorflow as tf
import random
import os

seed = 42
np.random.seed(seed)
tf.random.set_seed(seed)
random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)

# --- 1. Chargement des données ---
df = pd.read_csv("Données_Diff.csv", parse_dates=True, index_col=0)
df = df.sort_index()

# --- Nettoyage AVANT normalisation ---
df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=df.columns)  # supprime toutes les lignes avec NaN/Inf

# --- 2. Préparation des features et target ---
features = df.drop(columns=["result"])
target = df["result"]
y_binary = np.where(target == 1, 1, 0)

# --- 3. Normalisation ---
scaler_X = MinMaxScaler()
X_scaled = scaler_X.fit_transform(features)

# --- 4. Création des séquences ---
TIME_STEPS = 20
FUTURE_STEPS = 10

def create_sequences(X, y, time_steps):
    Xs, ys = [], []
    for i in range(len(X) - time_steps):
        Xs.append(X[i:i+time_steps])
        ys.append(y[i+time_steps])
    return np.array(Xs), np.array(ys)

X_seq, y_seq = create_sequences(X_scaled, y_binary, TIME_STEPS)
X_train, y_train = X_seq[:-FUTURE_STEPS], y_seq[:-FUTURE_STEPS]

# --- 5. Modèle GRU ---
model = Sequential()
model.add(Input(shape=(TIME_STEPS, X_seq.shape[2])))
model.add(GRU(64))
model.add(Dense(1, activation='sigmoid'))
model.compile(optimizer=Adam(learning_rate=0.0005), loss='binary_crossentropy', metrics=['accuracy'])


# --- 6. Entraînement ---
model.fit(X_seq, y_train, epochs=50, batch_size=32, validation_split=0.1, verbose=0)

# --- 7. Prédiction des 10 prochains jours ---
last_sequence = X_scaled[-TIME_STEPS:]
input_seq = last_sequence.copy()

probas = []
for _ in range(FUTURE_STEPS):
    input_batch = np.expand_dims(input_seq, axis=0)
    prob = model.predict(input_batch)[0][0]
    probas.append(prob)
    input_seq = np.roll(input_seq, -1, axis=0)
    input_seq[-1] = input_seq[-2]  # On répète la dernière ligne

# --- 8. Résultat ---
results_df = pd.DataFrame({
    "Jour": [f"J+{i+1}" for i in range(FUTURE_STEPS)],
    "P(hausse)": [round(p, 3) for p in probas],
    "P(baisse)": [round(1 - p, 3) for p in probas],
    "Prédiction": ["Hausse" if p > 0.5 else "Baisse" for p in probas]
})
print(results_df)



1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 133ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
   Jour  P(hausse)  P(baisse) Prédiction
0   J+1      0.291      0.709     Baisse
1   J+2      0.655      0.345     Hausse
2   J+3      0.659      0.341     Hausse
3   J+4      0.593      0.407     Hausse
4   J+5      0.450      0.550     Baisse
5   J+6      0.347      0.653     Baisse
6   J+7      0.378      0.622     Baisse
7   J+8      0.510      0.490     Hausse
8   J+9      0.639      0.361     Hausse
9  J+10      0.671      0.329     Hausse


Modèle Gru avec précision sur les 10 derniers jour du CSV

In [47]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Input
from tabulate import tabulate
import tensorflow as tf
import random
import os

somme_hausse = []

seed = 3
np.random.seed(seed)
tf.random.set_seed(seed)
random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)

# --- 1. Chargement des données ---
df = pd.read_csv("Données_Diff.csv", parse_dates=True, index_col=0)
df = df.sort_index()

# --- Nettoyage AVANT normalisation ---
df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=df.columns)  # supprime toutes les lignes avec NaN/Inf

# --- 2. Préparation des features et target ---
features = df.drop(columns=["result"])
target = df["result"]
y_binary = np.where(target == 1, 1, 0)

# --- 3. Normalisation ---
scaler_X = MinMaxScaler()
X_scaled = scaler_X.fit_transform(features)

# --- 4. Création des séquences ---
TIME_STEPS = 20
FUTURE_STEPS = 5

def create_sequences(X, y, time_steps):
    Xs, ys = [], []
    for i in range(len(X) - time_steps):
        Xs.append(X[i:i+time_steps])
        ys.append(y[i+time_steps])
    return np.array(Xs), np.array(ys)

X_seq, y_seq = create_sequences(X_scaled, y_binary, TIME_STEPS)
X_train, y_train = X_seq[:-FUTURE_STEPS], y_seq[:-FUTURE_STEPS]

# --- 5. Modèle GRU ---
model = Sequential()
model.add(Input(shape=(TIME_STEPS, X_train.shape[2])))
model.add(GRU(64))
model.add(Dense(1, activation='sigmoid'))
model.compile(optimizer=Adam(learning_rate=0.0005), loss='binary_crossentropy', metrics=['accuracy'])

# --- 6. Entraînement ---
model.fit(X_train, y_train, epochs=50, batch_size=32, validation_split=0.2, verbose=0)

# --- 10. Prédiction sur les 10 derniers jours réels du CSV ---
X_test_last10 = X_seq[-FUTURE_STEPS:]  # Les 10 dernières séquences du dataset
y_test_last10 = y_seq[-FUTURE_STEPS:]  # Les vraies valeurs (0 ou 1) correspondantes

probas_last10 = model.predict(X_test_last10).flatten()
preds_last10 = (probas_last10 > 0.5).astype(int)

# On récupère les index des 10 derniers jours dans le DataFrame d'origine
idx_last10 = df.index[-FUTURE_STEPS:]
diff_jour_last10 = df.loc[idx_last10, "Diff_jour"].values

somme = diff_jour_last10[probas_last10 > 0.5].sum() 
somme_hausse.append(somme)

# Construction du tableau de comparaison
compare_df = pd.DataFrame({
    "Jour": [f"J-{FUTURE_STEPS-i}" for i in range(FUTURE_STEPS, 0, -1)],
    "P(hausse)": np.round(probas_last10, 3),
    "P(baisse)": np.round(1 - probas_last10, 3),
    "Prédiction": ["Hausse" if p else "Baisse" for p in preds_last10],
    "Réalité": ["Hausse" if y else "Baisse" for y in y_test_last10]
})

from tabulate import tabulate
print("\n📊 Comparaison sur les 10 derniers jours réels du CSV :")
print(tabulate(compare_df, headers='keys', tablefmt='github', showindex=False))

# Calcul de la précision sur les 10 derniers jours
accuracy = np.mean(preds_last10 == y_test_last10)

# Précision uniquement sur les cas où le modèle prédit une hausse
mask_hausse = preds_last10 == 1
if np.any(mask_hausse):
    accuracy_hausse = np.mean(preds_last10[mask_hausse] == y_test_last10[mask_hausse])
else:
    accuracy_hausse = np.nan  # Aucun cas de hausse prédit

print(f"\n Précision du modèle sur les 10 derniers jours : {accuracy:.2%}")
print(f" Précision des hausses sur les 10 derniers jours : {accuracy_hausse:.2%}")
print(f" Somme Diff_jour (prédiction Hausse > 0.??) : {somme:.3f}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step

📊 Comparaison sur les 10 derniers jours réels du CSV :
| Jour   |   P(hausse) |   P(baisse) | Prédiction   | Réalité   |
|--------|-------------|-------------|--------------|-----------|
| J-0    |       0.852 |       0.148 | Hausse       | Hausse    |
| J-1    |       0.324 |       0.676 | Baisse       | Baisse    |
| J-2    |       0.486 |       0.514 | Baisse       | Hausse    |
| J-3    |       0.084 |       0.916 | Baisse       | Baisse    |
| J-4    |       0.781 |       0.219 | Hausse       | Hausse    |

 Précision du modèle sur les 10 derniers jours : 80.00%
 Précision des hausses sur les 10 derniers jours : 100.00%
 Somme Diff_jour (prédiction Hausse > 0.??) : 216.260


Test de plusieurs seed + Calcul  du résultat des jours prédit hausse

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Input
from tabulate import tabulate
import tensorflow as tf
import random
import os

#seeds = [1, 2, 3, 4, 5]
seeds = random.sample(range(1, 1001), 5)
precisions = []
somme_hausse = []

# --- 1. Chargement des données ---
df = pd.read_csv("Données_Diff.csv", parse_dates=True, index_col=0)
df = df.sort_index()

# --- Nettoyage AVANT normalisation ---
df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=df.columns)  # supprime toutes les lignes avec NaN/Inf

# --- 2. Préparation des features et target ---
features = df.drop(columns=["result"])
target = df["result"]
y_binary = np.where(target == 1, 1, 0)

# --- 3. Normalisation ---
scaler_X = MinMaxScaler()
X_scaled = scaler_X.fit_transform(features)

# --- 4. Création des séquences ---
TIME_STEPS = 20 # Nombre de jours pour la séquence
FUTURE_STEPS = 3 # Nombre de jours à prédire

def create_sequences(X, y, time_steps):
    Xs, ys = [], []
    for i in range(len(X) - time_steps):
        Xs.append(X[i:i+time_steps])
        ys.append(y[i+time_steps])
    return np.array(Xs), np.array(ys)

X_seq, y_seq = create_sequences(X_scaled, y_binary, TIME_STEPS)
X_train, y_train = X_seq[:-FUTURE_STEPS], y_seq[:-FUTURE_STEPS]

# --- 5. Entraînement et évaluation du modèle pour chaque seed ---
for seed in seeds:
    # Fixer les seeds pour la reproductibilité
    np.random.seed(seed)
    tf.random.set_seed(seed)
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

    # --- 5. Modèle GRU ---
    model = Sequential()
    model.add(Input(shape=(TIME_STEPS, X_train.shape[2])))
    model.add(GRU(64))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer=Adam(learning_rate=0.0005), loss='binary_crossentropy', metrics=['accuracy'])


    # --- 6. Entraînement ---
    model.fit(X_train, y_train, epochs=50, batch_size=32, validation_split=0.2, verbose=0)

    # --- 10. Prédiction sur les 10 derniers jours réels du CSV ---
    X_test_last10 = X_seq[-FUTURE_STEPS:]
    y_test_last10 = y_seq[-FUTURE_STEPS:]

    probas_last10 = model.predict(X_test_last10).flatten()
    preds_last10 = (probas_last10 > 0.5).astype(int)

    accuracy = np.mean(preds_last10 == y_test_last10)
    precisions.append(accuracy)

    # --- Somme de Diff_jour pour les prédictions "Hausse" ---
    probas_last10 = model.predict(X_test_last10).flatten()
    preds_last10 = (probas_last10 > 0.5).astype(int)

    accuracy = np.mean(preds_last10 == y_test_last10)
    precisions.append(accuracy)

    # On récupère les index des 10 derniers jours dans le DataFrame d'origine
    idx_last10 = df.index[-FUTURE_STEPS:]
    diff_jour_last10 = df.loc[idx_last10, "Diff_jour"].values

    somme = diff_jour_last10[preds_last10 == 1].sum()
    somme_hausse.append(somme)

Moyenne = np.mean(somme_hausse)

# Affichage des résultats
print("\nPrécision du modèle sur les 10 derniers jours pour chaque seed :")
for s, acc, somme in zip(seeds, precisions, somme_hausse):
    print(f"Seed {s} : {acc:.2%} | Somme Diff_jour (prédiction Hausse) : {somme:.3f}")

print(f"Moyenne globale Diff_jour : {Moyenne:.3f}")

# test avec la F1 score 



1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step

Précision du modèle sur les 10 derniers jours pour chaque seed :
Seed 1 : 100.00% | Somme Diff_jour (prédiction Hausse) : 599.420
Seed 2 : 100.00% | Somme Diff_jour (prédiction Hausse) : 599.420
Seed 3 : 100.00% | Somme Diff_jour (prédiction Hausse) : 491.620
Seed 4 : 100.00% | Somme Diff_jour (prédiction Hausse) : 599.420
Seed 5 : 66.67% | Somme Diff_jour (prédiction Hausse) : 107.800
Moyenne globale Diff_jour : 479.536


Pour la prédiction de 10 lignes nimporte où dans les 20% plus récentes 

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Input
import tensorflow as tf
import random
import os
from IPython.display import display


# --- 1. Chargement des données ---
df = pd.read_csv("Données_Diff.csv", parse_dates=True, index_col=0)
df = df.sort_index()

# --- Nettoyage AVANT normalisation ---
df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=df.columns)  # supprime toutes les lignes avec NaN/Inf

# --- 2. Préparation des features et target ---
features = df.drop(columns=["result"])
target = df["result"]
y_binary = np.where(target == 1, 1, 0)

# --- 3. Normalisation ---
scaler_X = MinMaxScaler()
X_scaled = scaler_X.fit_transform(features)

# --- 4. paramètres ---
TIME_STEPS = 20
FUTURE_STEPS = 3
seeds = random.sample(range(1, 1001), 20)
seuil = 0.5


def create_sequences(X, y, time_steps):
    Xs, ys = [], []
    for i in range(len(X) - time_steps):
        Xs.append(X[i:i+time_steps])
        ys.append(y[i+time_steps])
    return np.array(Xs), np.array(ys)

X_seq, y_seq = create_sequences(X_scaled, y_binary, TIME_STEPS)


all_precisions = []
all_sommes = []
all_precisions_hausse = []

    
for seed in seeds:
    np.random.seed(seed + repeat*100)  # pour varier à chaque répétition
    tf.random.set_seed(seed + repeat*100)
    random.seed(seed + repeat*100)
    os.environ['PYTHONHASHSEED'] = str(seed + repeat*100)

    model = Sequential()
    model.add(Input(shape=(TIME_STEPS, X_seq.shape[2])))
    model.add(GRU(64))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer=Adam(learning_rate=0.0005), loss='binary_crossentropy', metrics=['accuracy'])
    model.fit(X_seq, y_seq, epochs=50, batch_size=32, validation_split=0.2, verbose=0)

    # Sélection aléatoire dans les 20% les plus récentes
    nb_seq = len(X_seq)
    start_20p = int(nb_seq * 0.8)
    end_20p = nb_seq - FUTURE_STEPS
    random_start = random.randint(start_20p, end_20p)
    X_test_rand10 = X_seq[random_start:random_start+FUTURE_STEPS]
    y_test_rand10 = y_seq[random_start:random_start+FUTURE_STEPS]

    probas_rand10 = model.predict(X_test_rand10, verbose=0).flatten()
    preds_rand10 = (probas_rand10 > 0.5).astype(int)

    accuracy = np.mean(preds_rand10 == y_test_rand10)
    all_precisions.append(accuracy)
    
    # Précision uniquement sur les cas où le modèle prédit une hausse
    mask_hausse = preds_rand10 == 1
    if np.any(mask_hausse):
        accuracy_hausse = np.mean(preds_rand10[mask_hausse] == y_test_rand10[mask_hausse])
    else:
        accuracy_hausse = 1  # Aucun cas de hausse prédit
    all_precisions_hausse.append(accuracy_hausse)


    # calcul de la somme de Diff_jour pour les prédictions "Hausse"
    # considère que l'on prend le trade de l'ouverture à la fermeture
    idx_rand10 = df.index[random_start+TIME_STEPS:random_start+TIME_STEPS+FUTURE_STEPS]
    diff_jour_rand10 = df.loc[idx_rand10, "Diff_jour"].values
    somme = diff_jour_rand10[probas_rand10 > seuil].sum()
    all_sommes.append(somme)

# Moyennes et sommes globales
mean_precision = np.mean(all_precisions)
somme_somme = np.sum(all_sommes)
precisions_percent = pd.Series([f"{acc:.2%}" for acc in all_precisions])
mean_precisions_hausse = np.mean(all_precisions_hausse)
precisions_hausse_percent = pd.Series([f"{acc3:.2%}" for acc3 in all_precisions_hausse])

print("Nombre de jours prédit    :", FUTURE_STEPS)
print("Nombre de modèles         :",len(seeds))
print("Seuil pour le trade       :", seuil)
print(f"Précision sur les hausses : {mean_precisions_hausse:.2%}")
print(f"Précision des modèles     : {mean_precision:.2%}")
print(f"Somme Diff_jour (Hausse)  : {somme_somme:.3f} points")

print(f"\nPrécision des modèles : ")
print(precisions_percent)
print(f"\nPrécision des hausses : ")
print(precisions_hausse_percent)

display(precisions_hausse_percent)  # Affiche le DataFrame complet dans une zone scrollab
display(precisions_percent)

Nombre de jours prédit    : 3
Nombre de modèles         : 20
Seuil pour le trade       : 0.5
Précision sur les hausses : 60.00%
Précision des modèles     : 48.33%
Somme Diff_jour (Hausse)  : -293.819 points

Précision des modèles : 
0       0.00%
1      66.67%
2       0.00%
3     100.00%
4     100.00%
5       0.00%
6      33.33%
7      66.67%
8      33.33%
9      33.33%
10     66.67%
11     66.67%
12     66.67%
13     66.67%
14      0.00%
15     33.33%
16     66.67%
17     66.67%
18     66.67%
19     33.33%
dtype: object

Précision des hausses : 
0       0.00%
1     100.00%
2       0.00%
3     100.00%
4     100.00%
5       0.00%
6      33.33%
7     100.00%
8      33.33%
9       0.00%
10    100.00%
11     66.67%
12    100.00%
13     50.00%
14    100.00%
15     50.00%
16     66.67%
17    100.00%
18     66.67%
19     33.33%
dtype: object


Utilisation d'un seul model 

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Input
from tensorflow.keras.models import load_model
import tensorflow as tf
import random
import os
from IPython.display import display


# --- 1. Chargement des données ---
df = pd.read_csv("Données_Diff.csv", parse_dates=True, index_col=0)
df = df.sort_index()

# --- Nettoyage AVANT normalisation ---
df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=df.columns)  # supprime toutes les lignes avec NaN/Inf

# --- 2. Préparation des features et target ---
features = df.drop(columns=["result"])
target = df["result"]
y_binary = np.where(target == 1, 1, 0)

# --- 3. Normalisation ---
scaler_X = MinMaxScaler()
X_scaled = scaler_X.fit_transform(features)

# --- 4. paramètres ---
TIME_STEPS = 20
FUTURE_STEPS = 3
seeds = random.sample(range(1, 1001), 5)
seuil = 0.5


def create_sequences(X, y, time_steps):
    Xs, ys = [], []
    for i in range(len(X) - time_steps):
        Xs.append(X[i:i+time_steps])
        ys.append(y[i+time_steps])
    return np.array(Xs), np.array(ys)

X_seq, y_seq = create_sequences(X_scaled, y_binary, TIME_STEPS)


all_precisions = []
all_sommes = []
all_precisions_hausse = []

model = Sequential()
model.add(Input(shape=(TIME_STEPS, X_seq.shape[2])))
model.add(GRU(64))
model.add(Dense(1, activation='sigmoid'))
model.compile(optimizer=Adam(learning_rate=0.0005), loss='binary_crossentropy', metrics=['accuracy'])
model.fit(X_seq, y_seq, epochs=50, batch_size=32, validation_split=0.2, verbose=0)

model.save("Model_GRU_1.keras")

loss, acc = model.evaluate(X_test_rand10, y_test_rand10, verbose=0)


for seed in seeds:
    np.random.seed(seed)  # pour varier à chaque répétition
    tf.random.set_seed(seed)
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

    model = load_model("Model_GRU_1.keras")

    # Sélection aléatoire dans les 20% les plus récentes
    nb_seq = len(X_seq)
    start_20p = int(nb_seq * 0.8)
    end_20p = nb_seq - FUTURE_STEPS
    random_start = random.randint(start_20p, end_20p)
    X_test_rand10 = X_seq[random_start:random_start+FUTURE_STEPS]
    y_test_rand10 = y_seq[random_start:random_start+FUTURE_STEPS]

    probas_rand10 = model.predict(X_test_rand10, verbose=0).flatten()
    preds_rand10 = (probas_rand10 > 0.5).astype(int)


    accuracy = np.mean(preds_rand10 == y_test_rand10)
    all_precisions.append(accuracy)
    
    # Précision uniquement sur les cas où le modèle prédit une hausse
    mask_hausse = preds_rand10 == 1
    if np.any(mask_hausse):
        accuracy_hausse = np.mean(preds_rand10[mask_hausse] == y_test_rand10[mask_hausse])
    else:
        accuracy_hausse = 1  # Aucun cas de hausse prédit
    all_precisions_hausse.append(accuracy_hausse)


    # calcul de la somme de Diff_jour pour les prédictions "Hausse"
    # considère que l'on prend le trade de l'ouverture à la fermeture
    idx_rand10 = df.index[random_start+TIME_STEPS:random_start+TIME_STEPS+FUTURE_STEPS]
    diff_jour_rand10 = df.loc[idx_rand10, "Diff_jour"].values
    somme = diff_jour_rand10[probas_rand10 > seuil].sum()
    all_sommes.append(somme)

# Moyennes et sommes globales
mean_precision = np.mean(all_precisions)
somme_somme = np.sum(all_sommes)
precisions_percent = pd.Series([f"{acc:.2%}" for acc in all_precisions])
mean_precisions_hausse = np.mean(all_precisions_hausse)
precisions_hausse_percent = pd.Series([f"{acc3:.2%}" for acc3 in all_precisions_hausse])

print("Nombre de jours prédit    :", FUTURE_STEPS)
print("Nombre de modèles         :",len(seeds))
print("Seuil pour le trade       :", seuil)
print(f"Précision sur les hausses : {mean_precisions_hausse:.2%}")
print(f"Précision des modèles     : {mean_precision:.2%}")
print(f"Somme Diff_jour (Hausse)  : {somme_somme:.3f} points")

print("\nAccuracy:", acc)

print(f"\nPrécision des modèles : ")
display(precisions_percent)
print(f"\nPrécision des hausses : ")
display(precisions_hausse_percent)



Nombre de jours prédit    : 3
Nombre de modèles         : 5
Seuil pour le trade       : 0.5
Précision sur les hausses : 80.00%
Précision des modèles     : 33.33%
Somme Diff_jour (Hausse)  : -19.770 points

Accuracy: 0.3333333432674408

Précision des modèles : 


0      0.00%
1     33.33%
2    100.00%
3      0.00%
4     33.33%
dtype: object


Précision des hausses : 


0    100.00%
1    100.00%
2    100.00%
3    100.00%
4      0.00%
dtype: object